# Producer- Inference-Consumer

The logic must be in a function, and all inputs must be standard Python arguments (not environment variables).

No interactive input() or blocking user interaction.

You should use the kfp.components.create_component_from_func or build a Docker container with your code.

(All Args Explicit, No ENV)

In [1]:
#Global libraries

import kfp
from kfp import dsl
from typing import NamedTuple


In [9]:
############################################################################################################
#####################################---1. Producer Stage---################################################
############################################################################################################
@dsl.component(
    base_image="docker.io/jhofydu/pytorch-kfp:v1.0.0",
    packages_to_install=["minio", "prometheus-api-client", "kafka-python"]
)

def producer_intra():
    # %pip install prometheus-api-client kafka-python

    import os
    import time
    import json
    import datetime
    from typing import List
    from prometheus_api_client import PrometheusConnect
    from kafka import KafkaProducer
    
    def fresh_batches_and_publish(
        prom_url: str = os.getenv("PROM_URL", "http://prometheus-operated.prometheus.svc.cluster.local:9090"),
        kafka_bootstrap: str = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka.apache-kafka.svc.cluster.local:9092"),
        topic: str = os.getenv("KAFKA_TOPIC", "cpu-batch"),
        prom_query: str = os.getenv(
            "PROM_QUERY",
            'sum by (instance) (rate(node_cpu_seconds_total{mode!="idle"}[1m]))'
        ),
        target_instance: str = os.getenv("TARGET_INSTANCE", None),   # e.g. "172.19.0.3:9100"
        batch_size: int = int(os.getenv("BATCH_SIZE", "24")),
        interval_sec: int = int(os.getenv("INTERVAL_SEC", "10")),
        print_each_batch: bool = True,
    ):
        """
        Collect exactly batch_size points for each batch, publish, repeat. Start fresh every run.
        Runs infinitely!
        """
        prom = PrometheusConnect(url=prom_url, disable_ssl=True)
        producer = KafkaProducer(
            bootstrap_servers=[s.strip() for s in kafka_bootstrap.split(",") if s.strip()],
            value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        )
    
        print("============ CONFIG ============")
        print(f"Prometheus: {prom_url}")
        print(f"Query:      {prom_query}")
        print(f"Instance:   {target_instance if target_instance else 'ALL'}")
        print(f"Kafka:      {kafka_bootstrap}")
        print(f"Topic:      {topic}")
        print(f"Interval:   {interval_sec} sec")
        print(f"Batch size: {batch_size}")
        print("================================\n")
    
        batches_sent = 0
    
        while True:
            batch_points: List[dict] = []
            print(f"[producer] Collecting batch #{batches_sent+1} ...")
            while len(batch_points) < batch_size:
                tick_ts = int(time.time())
                try:
                    result = prom.custom_query(query=prom_query)
                except Exception as e:
                    print(f"[producer] Prometheus query FAILED: {e}")
                    time.sleep(interval_sec)
                    continue
    
                found = False
                for sample in result:
                    inst = sample["metric"].get("instance")
                    if target_instance and inst != target_instance:
                        continue
                    cpu_val = float(sample["value"][1])
                    ts_human = datetime.datetime.fromtimestamp(tick_ts).strftime("%Y-%m-%d %H:%M:%S")
                    print(f"[{len(batch_points)+1:02d}] {ts_human}, {cpu_val:.8f}")
                    batch_points.append({
                        "ts": ts_human,
                        "cpu_pct": cpu_val,
                    })
                    found = True
                    break  # only take first match for target_instance
    
                if not found:
                    print("[producer] No data for instance, retrying...")
                if len(batch_points) < batch_size:
                    time.sleep(interval_sec)  # wait for next point
    
            # Build message for Kafka
            msg = {
                "instance": target_instance,
                "batch_ts": [pt["ts"] for pt in batch_points],
                "batch_cpu_pct": [pt["cpu_pct"] for pt in batch_points],
                "batch_number": batches_sent + 1,
            }
            producer.send(topic, value=msg)
            producer.flush()
            if print_each_batch:
                print(f"\n[producer] Published batch #{batches_sent+1} for {target_instance}:")
                for i, (ts, v) in enumerate(zip(msg["batch_ts"], msg["batch_cpu_pct"]), 1):
                    print(f"  [{i:02d}] {ts}, {v:.8f}")
                print("-" * 40)
            batches_sent += 1
    
            # --- Fix: Wait for next *new* interval before starting the next batch ---
            time.sleep(interval_sec)
    
        # (never exits, runs forever)
    




##########################################################################################################
#####################################---Connecting Stages---################################################
############################################################################################################

@dsl.pipeline(name="kafka_intracom_pipeline", description="Generate a kafka pipeline to feed a bacthing forecasting model cpu% in k8s")
def kafka_intracom():
    caching_option = False

    step1_1 = producer_intra().set_caching_options(caching_option)
    
    

###########################################################################################
#####################################---Running Pipeline---################################
###########################################################################################
if __name__ == '__main__':
    import kfp
    client = kfp.Client()
    client.create_run_from_pipeline_func(
        kafka_intracom,
        arguments={},
        experiment_name="kafka-intracom-pipeline"
    )